In [2]:
import serial
%gui wx
import wx
import numpy as np



In [ ]:
class Gui(wx.Frame):
    CRC_TABLE = [0x00, 0xD5, 0x7F, 0xAA, 0xFE, 0x2B, 0x81, 0x54, 0x29, 0xFC, 0x56, 0x83, 0xD7, 0x02, 0xA8, 0x7D,
	          0x52, 0x87, 0x2D, 0xF8, 0xAC, 0x79, 0xD3, 0x06, 0x7B, 0xAE, 0x04, 0xD1, 0x85, 0x50, 0xFA, 0x2F,
	          0xA4, 0x71, 0xDB, 0x0E, 0x5A, 0x8F, 0x25, 0xF0, 0x8D, 0x58, 0xF2, 0x27, 0x73, 0xA6, 0x0C, 0xD9,
	          0xF6, 0x23, 0x89, 0x5C, 0x08, 0xDD, 0x77, 0xA2, 0xDF, 0x0A, 0xA0, 0x75, 0x21, 0xF4, 0x5E, 0x8B,
	          0x9D, 0x48, 0xE2, 0x37, 0x63, 0xB6, 0x1C, 0xC9, 0xB4, 0x61, 0xCB, 0x1E, 0x4A, 0x9F, 0x35, 0xE0,
	          0xCF, 0x1A, 0xB0, 0x65, 0x31, 0xE4, 0x4E, 0x9B, 0xE6, 0x33, 0x99, 0x4C, 0x18, 0xCD, 0x67, 0xB2,
	          0x39, 0xEC, 0x46, 0x93, 0xC7, 0x12, 0xB8, 0x6D, 0x10, 0xC5, 0x6F, 0xBA, 0xEE, 0x3B, 0x91, 0x44,
	          0x6B, 0xBE, 0x14, 0xC1, 0x95, 0x40, 0xEA, 0x3F, 0x42, 0x97, 0x3D, 0xE8, 0xBC, 0x69, 0xC3, 0x16,
	          0xEF, 0x3A, 0x90, 0x45, 0x11, 0xC4, 0x6E, 0xBB, 0xC6, 0x13, 0xB9, 0x6C, 0x38, 0xED, 0x47, 0x92,
	          0xBD, 0x68, 0xC2, 0x17, 0x43, 0x96, 0x3C, 0xE9, 0x94, 0x41, 0xEB, 0x3E, 0x6A, 0xBF, 0x15, 0xC0,
	          0x4B, 0x9E, 0x34, 0xE1, 0xB5, 0x60, 0xCA, 0x1F, 0x62, 0xB7, 0x1D, 0xC8, 0x9C, 0x49, 0xE3, 0x36,
	          0x19, 0xCC, 0x66, 0xB3, 0xE7, 0x32, 0x98, 0x4D, 0x30, 0xE5, 0x4F, 0x9A, 0xCE, 0x1B, 0xB1, 0x64,
	          0x72, 0xA7, 0x0D, 0xD8, 0x8C, 0x59, 0xF3, 0x26, 0x5B, 0x8E, 0x24, 0xF1, 0xA5, 0x70, 0xDA, 0x0F,
	          0x20, 0xF5, 0x5F, 0x8A, 0xDE, 0x0B, 0xA1, 0x74, 0x09, 0xDC, 0x76, 0xA3, 0xF7, 0x22, 0x88, 0x5D,
	          0xD6, 0x03, 0xA9, 0x7C, 0x28, 0xFD, 0x57, 0x82, 0xFF, 0x2A, 0x80, 0x55, 0x01, 0xD4, 0x7E, 0xAB,
	          0x84, 0x51, 0xFB, 0x2E, 0x7A, 0xAF, 0x05, 0xD0, 0xAD, 0x78, 0xD2, 0x07, 0x53, 0x86, 0x2C, 0xF9]
    
    def __init__(self, title):
        """Initialise widgets and layout."""
        super().__init__(parent=None, title=title, size=(800, 400))
        self.panel = wx.Panel(self)
        
        self.rx_frame_ctr = 0
        self.rx_err_ctr = 0
        self.rx_type = {}

        # button_sizer = wx.StaticBoxSizer(wx.StaticBox(panel,label='Serial'),wx.VERTICAL)
        # button_sizer.Add(wx.Button(panel, label="Connect"),flag=wx.BOTTOM,border=5)
        self.gbs = wx.GridBagSizer(6,16)
        self.ch_label = []
        self.ch_value = []
        self.connect_button = wx.Button(self.panel,label='Connect')
        self.connect_button.Bind(wx.EVT_BUTTON, self.ConnectClick)
        self.gbs.Add(self.connect_button, pos=(0, 0),span=(0,4), flag=wx.EXPAND)
        for i in range(16):
            self.ch_label.append(wx.StaticText(self.panel, label=f'ch{i}'))
            self.ch_value.append(wx.StaticText(self.panel, label=f'{0:4}',style=wx.ALIGN_CENTRE_HORIZONTAL))
            self.gbs.Add(self.ch_label[i], pos=(1, i), flag=wx.EXPAND)
            self.gbs.Add(self.ch_value[i], pos=(2, i), flag=wx.EXPAND)
            # gbs.AddGrowableRow(i)
            self.gbs.AddGrowableCol(i)

        self.stats_label = wx.StaticText(self.panel,label ='frame count')
        self.stats_text = wx.StaticText(self.panel,label =f'{self.rx_frame_ctr} / {self.rx_err_ctr}')
        self.gbs.Add(self.stats_label, pos=(3, 0), flag=wx.EXPAND)
        self.gbs.Add(self.stats_text, pos=(3, 1),span=(0,4), flag=wx.EXPAND)

        self.type_label = wx.StaticText(self.panel,label ='frame type')
        self.type_text = wx.StaticText(self.panel,label =f'-')
        self.gbs.Add(self.type_label, pos=(4, 0), flag=wx.EXPAND)
        self.gbs.Add(self.type_text, pos=(4, 1),span=(0,12), flag=wx.EXPAND)

        # Panel should use the StaticBoxSizer to layout
        self.panel.SetSizer(self.gbs)

        self.ser = serial.Serial()
        self.ser.port = 'COM3'
        self.ser.baudrate = 416666
        self.ser.timeout = 1


        # Initialize wx.Timer and Bind event
        self.count = 0
        self.timer = wx.Timer(self)
        self.Bind(wx.EVT_TIMER, self.on_timer_tick, self.timer)
        self.Bind(wx.EVT_CLOSE, self.OnClose)
    def OnClose(self, event):
        if self.ser.is_open:
            try:
                self.ser.close()
                print('Serial Port Closed')
                self.connect_button.SetLabel('Connect')
            except serial.SerialException as e:
                print(f"Error closing port: {e}")
        self.timer.Stop()
        self.Destroy()

    def ConnectClick(self, event):
        if self.ser.is_open:
            try:
                self.ser.close()
                print('Serial Port Closed')
                self.connect_button.SetLabel('Connect')
            except serial.SerialException as e:
                print(f"Error closing port: {e}")
            self.timer.Stop()
        else:
            try:
                self.ser.open()
                print('Serial Port Opened')
                self.connect_button.SetLabel('Connected')
                self.ser.reset_input_buffer()
                self.timer.Start(100)
            except serial.SerialException as e:
                print(f"Error opening port: {e}")
            self.ch_value[2].SetLabel('2')


    def on_timer_tick(self, event):
        self.count += 1
        if self.ser.in_waiting > 64:
            self.rx_buffer = self.ser.read(self.ser.in_waiting)
            self.rx_buffer,msg_len,frame = self.get_frame(self.rx_buffer)
            self.rx_frame_ctr += 1
            if self.calculate_crc8(frame) != 0:
                self.rx_err_ctr += 1
                print(f'CRC Failed {hex(frame)}')
            else:
                frame_type = str(hex(frame[2]))
                print(frame)
                if frame_type in self.rx_type:
                    self.rx_type[frame_type] += 1
                else:
                    self.rx_type[frame_type] = 1
                if frame_type == '0x16':                    
                    print('Found')
                    rc_ch = self.parser_rc_channel(frame[2:])
                    
                    # print(rc_ch['ch1'])
                    # self.ch_value[0].SetLabel(str(rc_ch['ch1']))
                    # print(rc_ch)
                    ch = 0
                    for k in rc_ch:
                        self.ch_value[ch].SetLabel(str(rc_ch[k]) )
                        ch +=1
                    
        self.stats_text.SetLabel(f'{self.rx_frame_ctr} / {self.rx_err_ctr}')
        type_text = ""
        for k in self.rx_type:
            type_text += f'{k}:{self.rx_type[k]}, '
        self.type_text.SetLabel(type_text)
        # self.panel.Refresh()
        # self.panel.Update()
    
    def get_frame(self, rx_data:bytes):
        idx = rx_data.find(b'\xC8')
        if idx == -1:
            rx_data = []
        else:
            msg_len = rx_data[idx+1]
            print(msg_len)
            frame = rx_data[idx : idx + 2 + msg_len]
            rx_data = rx_data[idx + 2 + msg_len:]
        return rx_data,msg_len,frame

    def calculate_crc8(self, data: bytes, init=0x00):
        # data = bytes.fromhex(data_str)
        crc = init
        for byte in data[2:]:
            crc = crc ^ byte
            crc = self.CRC_TABLE[crc]
        return crc

    def parser_rc_channel(self,frame):
        us_dict = {}
        ticks_dict = {}
        ticks = int.from_bytes(frame[1:3],byteorder='big') >> 5
        ticks = ticks & 0x7FF
        ticks_dict['ch1'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch1'] = us

        ticks = int.from_bytes(frame[2:4]) >> 2
        ticks = ticks & 0x7FF
        ticks_dict['ch2'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch2'] = us

        ticks = int.from_bytes(frame[3:6]) >> 7
        ticks = ticks & 0x7FF
        ticks_dict['ch3'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch3'] = us

        ticks = int.from_bytes(frame[5:7]) >> 4
        ticks = ticks & 0x7FF
        ticks_dict['ch4'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch4'] = us

        ticks = int.from_bytes(frame[6:9]) >> 1
        ticks = ticks & 0x7FF
        ticks_dict['ch5'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch5'] = us

        ticks = int.from_bytes(frame[7:10]) >> 6
        ticks = ticks & 0x7FF
        ticks_dict['ch6'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch6'] = us

        ticks = int.from_bytes(frame[9:11]) >> 3
        ticks = ticks & 0x7FF
        ticks_dict['ch7'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch7'] = us

        ticks = int.from_bytes(frame[10:12]) >> 0
        ticks = ticks & 0x7FF
        ticks_dict['ch8'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch8'] = us

        ticks = int.from_bytes(frame[12:14]) >> 5
        ticks = ticks & 0x7FF
        ticks_dict['ch9'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch9'] = us

        ticks = int.from_bytes(frame[13:15]) >> 2
        ticks = ticks & 0x7FF
        ticks_dict['ch10'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch10'] = us

        ticks = int.from_bytes(frame[14:17]) >> 7
        ticks = ticks & 0x7FF
        ticks_dict['ch11'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch11'] = us

        ticks = int.from_bytes(frame[16:18]) >> 4
        ticks = ticks & 0x7FF
        ticks_dict['ch12'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch12'] = us

        ticks = int.from_bytes(frame[17:20]) >> 1
        ticks = ticks & 0x7FF
        ticks_dict['ch13'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch13'] = us

        ticks = int.from_bytes(frame[18:21]) >> 6
        ticks = ticks & 0x7FF
        ticks_dict['ch14'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch14'] = us

        ticks = int.from_bytes(frame[20:22]) >> 3
        ticks = ticks & 0x7FF
        ticks_dict['ch15'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch15'] = us

        ticks = int.from_bytes(frame[21:23]) >> 0
        ticks = ticks & 0x7FF
        ticks_dict['ch16'] = ticks
        us = ((ticks - 992) * 5 / 8 + 1500)
        us_dict['ch16'] = us
        return us_dict
    
app = wx.GetApp() or wx.App(False)
gui = Gui("ERLS")
gui.Show(True)
# app.MainLoop()

True

Serial Port Opened
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
12
0x14
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
24
0x16
Found
12
0x14
12
0x14
12
0x14
24
0x16
Found
24
0x16
Found

In [ ]:

class MyFrame(wx.Frame):
    def __init__(self, *args, **kw):
        super(MyFrame, self).__init__(*args, **kw)
        panel = wx.Panel(self)
        
        # 1. Create the Static Box
        static_box = wx.StaticBox(panel, label="User Information")
        
        # 2. Create the GridBagSizer
        gbs = wx.GridBagSizer(vgap=10, hgap=10)
        
        # 3. Nest the sizer in the static box
        static_box.SetSizer(gbs)
        
        # 4. Add Widgets to the GridBagSizer
        gbs.Add(wx.StaticText(static_box, label="Name:"), pos=(0, 0), flag=wx.ALIGN_CENTER_VERTICAL)
        gbs.Add(wx.TextCtrl(static_box), pos=(0, 1), flag=wx.EXPAND)
        
        gbs.Add(wx.StaticText(static_box, label="Age:"), pos=(1, 0), flag=wx.ALIGN_CENTER_VERTICAL)
        gbs.Add(wx.TextCtrl(static_box), pos=(1, 1), flag=wx.EXPAND)
        
        # Make the second column expandable
        gbs.AddGrowableCol(1)
        
        # Wrap static box in a main layout to control its size on the panel
        main_sizer = wx.BoxSizer(wx.VERTICAL)
        main_sizer.Add(static_box, proportion=0, flag=wx.ALL|wx.EXPAND, border=15)
        panel.SetSizer(main_sizer)

app = wx.App()
frame = MyFrame(None, title="GridBagSizer with StaticBox", size=(400, 250))
frame.Show()

In [ ]:
class MyFrame(wx.Frame):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        panel = wx.Panel(self)
        
        gbs = wx.GridBagSizer(5, 5)
        gbs.Add(wx.Button(panel, label="Button 1"), pos=(0, 0))
        gbs.Add(wx.Button(panel, label="Button 2"), pos=(0, 1))

        # Create a StaticBox around the panel to serve as a visible border
        sb = wx.StaticBox(panel, label="Grid Bag Border")
        box_sizer = wx.StaticBoxSizer(sb, wx.VERTICAL)
        
        # Add the GridBagSizer to the StaticBoxSizer
        box_sizer.Add(gbs, flag=wx.ALL | wx.EXPAND, border=15)
        
        # Panel should use the StaticBoxSizer to layout
        panel.SetSizer(box_sizer)
app = []
app = wx.GetApp() or wx.App(False)
frame = MyFrame(None, title="Visible Border Example", size=(400, 300))
frame.Show()

In [ ]:


frame = wx.Frame(None, title="Hello from Jupyter")
frame.Show()

In [ ]:
import serial
import time
import argparse
from enum import IntEnum

CRSF_SYNC = 0xC8

class PacketsTypes(IntEnum):
    GPS = 0x02
    VARIO = 0x07
    BATTERY_SENSOR = 0x08
    BARO_ALT = 0x09
    HEARTBEAT = 0x0B
    VIDEO_TRANSMITTER = 0x0F
    LINK_STATISTICS = 0x14
    RC_CHANNELS_PACKED = 0x16
    ATTITUDE = 0x1E
    FLIGHT_MODE = 0x21
    DEVICE_INFO = 0x29
    CONFIG_READ = 0x2C
    CONFIG_WRITE = 0x2D
    RADIO_ID = 0x3A

def crc8_dvb_s2(crc, a) -> int:
  crc = crc ^ a
  for ii in range(8):
    if crc & 0x80:
      crc = (crc << 1) ^ 0xD5
    else:
      crc = crc << 1
  return crc & 0xFF

def crc8_data(data) -> int:
    crc = 0
    for a in data:
        crc = crc8_dvb_s2(crc, a)
    return crc

def crsf_validate_frame(frame) -> bool:
    return crc8_data(frame[2:-1]) == frame[-1]

def signed_byte(b):
    return b - 256 if b >= 128 else b

def packCrsfToBytes(channels) -> bytes:
    # channels is in CRSF format! (0-1984)
    # Values are packed little-endianish such that bits BA987654321 -> 87654321, 00000BA9
    # 11 bits per channel x 16 channels = 22 bytes
    if len(channels) != 16:
        raise ValueError('CRSF must have 16 channels')
    result = bytearray()
    destShift = 0
    newVal = 0
    for ch in channels:
        # Put the low bits in any remaining dest capacity
        newVal |= (ch << destShift) & 0xff
        result.append(newVal)

        # Shift the high bits down and place them into the next dest byte
        srcBitsLeft = 11 - 8 + destShift
        newVal = ch >> (11 - srcBitsLeft)
        # When there's at least a full byte remaining, consume that as well
        if srcBitsLeft >= 8:
            result.append(newVal & 0xff)
            newVal >>= 8
            srcBitsLeft -= 8

        # Next dest should be shifted up by the bits consumed
        destShift = srcBitsLeft

    return result

def channelsCrsfToChannelsPacket(channels) -> bytes:
    result = bytearray([CRSF_SYNC, 24, PacketsTypes.RC_CHANNELS_PACKED]) # 24 is packet length
    result += packCrsfToBytes(channels)
    result.append(crc8_data(result[2:]))
    return result

def handleCrsfPacket(ptype, data):
    if ptype == PacketsTypes.RADIO_ID and data[5] == 0x10:
        #print(f"OTX sync")
        pass
    elif ptype == PacketsTypes.LINK_STATISTICS:
        rssi1 = signed_byte(data[3])
        rssi2 = signed_byte(data[4])
        lq = data[5]
        snr = signed_byte(data[6])
        antenna = data[7]
        mode = data[8]
        power = data[9]
        # telemetry strength
        downlink_rssi = signed_byte(data[10])
        downlink_lq = data[11]
        downlink_snr = signed_byte(data[12])
        print(f"RSSI={rssi1}/{rssi2}dBm LQ={lq:03} mode={mode}") # ant={antenna} snr={snr} power={power} drssi={downlink_rssi} dlq={downlink_lq} dsnr={downlink_snr}")
    elif ptype == PacketsTypes.ATTITUDE:
        pitch = int.from_bytes(data[3:5], byteorder='big', signed=True) / 10000.0
        roll = int.from_bytes(data[5:7], byteorder='big', signed=True) / 10000.0
        yaw = int.from_bytes(data[7:9], byteorder='big', signed=True) / 10000.0
        print(f"Attitude: Pitch={pitch:0.2f} Roll={roll:0.2f} Yaw={yaw:0.2f} (rad)")
    elif ptype == PacketsTypes.FLIGHT_MODE:
        packet = ''.join(map(chr, data[3:-2]))
        print(f"Flight Mode: {packet}")
    elif ptype == PacketsTypes.BATTERY_SENSOR:
        vbat = int.from_bytes(data[3:5], byteorder='big', signed=True) / 10.0
        curr = int.from_bytes(data[5:7], byteorder='big', signed=True) / 10.0
        mah = data[7] << 16 | data[8] << 7 | data[9]
        pct = data[10]
        print(f"Battery: {vbat:0.2f}V {curr:0.1f}A {mah}mAh {pct}%")
    elif ptype == PacketsTypes.BARO_ALT:
        print(f"BaroAlt: ")
    elif ptype == PacketsTypes.DEVICE_INFO:
        packet = ' '.join(map(hex, data))
        print(f"Device Info: {packet}")
    elif data[2] == PacketsTypes.GPS:
        lat = int.from_bytes(data[3:7], byteorder='big', signed=True) / 1e7
        lon = int.from_bytes(data[7:11], byteorder='big', signed=True) / 1e7
        gspd = int.from_bytes(data[11:13], byteorder='big', signed=True) / 36.0
        hdg =  int.from_bytes(data[13:15], byteorder='big', signed=True) / 100.0
        alt = int.from_bytes(data[15:17], byteorder='big', signed=True) - 1000
        sats = data[17]
        print(f"GPS: Pos={lat} {lon} GSpd={gspd:0.1f}m/s Hdg={hdg:0.1f} Alt={alt}m Sats={sats}")
    elif ptype == PacketsTypes.VARIO:
        vspd = int.from_bytes(data[3:5], byteorder='big', signed=True) / 10.0
        print(f"VSpd: {vspd:0.1f}m/s")
    elif ptype == PacketsTypes.RC_CHANNELS_PACKED:
        #print(f"Channels: (data)")
        pass
    else:
        packet = ' '.join(map(hex, data))
        print(f"Unknown 0x{ptype:02x}: {packet}")


In [ ]:
crsf_validate_frame(b'c81816e0035fc3c107f05ffc02e0bf0078f9ca0700004c7ce24e')

In [ ]:
a = crc8_data(b'16')
print(a)

In [ ]:
crc  = 0
crc = crc8_dvb_s2(crc, int(b'16'))

In [ ]:
def get_frame(rx_data:bytes):
    idx = rx_data.find(b'\xC8')
    print(idx)
    if idx == -1:
        rx_data = []
    else:
        msg_len = rx_data[idx+1]
        print(msg_len)
        frame = rx_data[idx + 2: idx + 2 + msg_len]
        rx_data = rx_data[idx + 2 + msg_len:]
    return rx_data,msg_len,frame

def calculate_crc8(data: bytes, init=0x00):
    # data = bytes.fromhex(data_str)
    crc = init
    for byte in data:
        crc = crc ^ byte
        crc = crc_table[crc]
    return crc

crc_table = [0x00, 0xD5, 0x7F, 0xAA, 0xFE, 0x2B, 0x81, 0x54, 0x29, 0xFC, 0x56, 0x83, 0xD7, 0x02, 0xA8, 0x7D,
	          0x52, 0x87, 0x2D, 0xF8, 0xAC, 0x79, 0xD3, 0x06, 0x7B, 0xAE, 0x04, 0xD1, 0x85, 0x50, 0xFA, 0x2F,
	          0xA4, 0x71, 0xDB, 0x0E, 0x5A, 0x8F, 0x25, 0xF0, 0x8D, 0x58, 0xF2, 0x27, 0x73, 0xA6, 0x0C, 0xD9,
	          0xF6, 0x23, 0x89, 0x5C, 0x08, 0xDD, 0x77, 0xA2, 0xDF, 0x0A, 0xA0, 0x75, 0x21, 0xF4, 0x5E, 0x8B,
	          0x9D, 0x48, 0xE2, 0x37, 0x63, 0xB6, 0x1C, 0xC9, 0xB4, 0x61, 0xCB, 0x1E, 0x4A, 0x9F, 0x35, 0xE0,
	          0xCF, 0x1A, 0xB0, 0x65, 0x31, 0xE4, 0x4E, 0x9B, 0xE6, 0x33, 0x99, 0x4C, 0x18, 0xCD, 0x67, 0xB2,
	          0x39, 0xEC, 0x46, 0x93, 0xC7, 0x12, 0xB8, 0x6D, 0x10, 0xC5, 0x6F, 0xBA, 0xEE, 0x3B, 0x91, 0x44,
	          0x6B, 0xBE, 0x14, 0xC1, 0x95, 0x40, 0xEA, 0x3F, 0x42, 0x97, 0x3D, 0xE8, 0xBC, 0x69, 0xC3, 0x16,
	          0xEF, 0x3A, 0x90, 0x45, 0x11, 0xC4, 0x6E, 0xBB, 0xC6, 0x13, 0xB9, 0x6C, 0x38, 0xED, 0x47, 0x92,
	          0xBD, 0x68, 0xC2, 0x17, 0x43, 0x96, 0x3C, 0xE9, 0x94, 0x41, 0xEB, 0x3E, 0x6A, 0xBF, 0x15, 0xC0,
	          0x4B, 0x9E, 0x34, 0xE1, 0xB5, 0x60, 0xCA, 0x1F, 0x62, 0xB7, 0x1D, 0xC8, 0x9C, 0x49, 0xE3, 0x36,
	          0x19, 0xCC, 0x66, 0xB3, 0xE7, 0x32, 0x98, 0x4D, 0x30, 0xE5, 0x4F, 0x9A, 0xCE, 0x1B, 0xB1, 0x64,
	          0x72, 0xA7, 0x0D, 0xD8, 0x8C, 0x59, 0xF3, 0x26, 0x5B, 0x8E, 0x24, 0xF1, 0xA5, 0x70, 0xDA, 0x0F,
	          0x20, 0xF5, 0x5F, 0x8A, 0xDE, 0x0B, 0xA1, 0x74, 0x09, 0xDC, 0x76, 0xA3, 0xF7, 0x22, 0x88, 0x5D,
	          0xD6, 0x03, 0xA9, 0x7C, 0x28, 0xFD, 0x57, 0x82, 0xFF, 0x2A, 0x80, 0x55, 0x01, 0xD4, 0x7E, 0xAB,
	          0x84, 0x51, 0xFB, 0x2E, 0x7A, 0xAF, 0x05, 0xD0, 0xAD, 0x78, 0xD2, 0x07, 0x53, 0x86, 0x2C, 0xF9]

In [ ]:
rx_data = 'aac81816e003dfe6c007f05ffc02e0bf0078f9ca0700004c7ce21cc81816e003dfe6c007f05ffc02e0bf0078f9ca0700004c7ce21cc81816e003dfe6c007f05'

In [ ]:
rx_data = 'aac81816e003dfe6c007f05ffc02e0bf0078f9ca0700004c7ce21cc8'
data = bytes.fromhex(rx_data)
data,msg_len,frame = get_frame(data)
calculate_crc8(frame)
hex(frame[0])

In [ ]:
def parser_rc_channel(frame):
    ch1 = frame[1:3]

    us_dict = {}
    us_dict['ch1'] = 
    return us_dict

rx_rc = parser_rc_channel(frame)
np.frombuffer(rx_rc[0:2],dtype=np.uint16)

#define TICKS_TO_US(x)  ((x - 992) * 5 / 8 + 1500)
#define US_TO_TICKS(x)  ((x - 1500) * 8 / 5 + 992)

In [ ]:
ticks = int.from_bytes(rx_rc) >> 5
us = ((ticks - 992) * 5 / 8 + 1500)
print(us)